In [0]:
%pip install openpyxl
dbutils.library.restartPython()

In [0]:
import requests, urllib3, os, json
from datetime import datetime
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

CATALOGO = "mvp_pipeline"
VOL = f"/Volumes/{CATALOGO}/bronze/arquivos_brutos"

# (tabela_destino, subpasta, nome_arquivo, url)
FONTES_CSV = [
    ("icms_cnae_subclasse", "sefaz", "arrecadacao_icms_cnae_subclasse.csv",
     "https://receitadados.sefaz.rs.gov.br/Arquivos/Arrecada%C3%A7%C3%A3o%20de%20ICMS%20por%20CNAE%20-%20Subclasse.csv"),
    ("arrecadacao_municipio_corede", "sefaz", "arrecadacao_municipio_corede.csv",
     "https://receitadados.sefaz.rs.gov.br/Arquivos/Arrecada%C3%A7%C3%A3o%20por%20Munic%C3%ADpio%20e%20por%20Corede.csv"),
    ("desoneracoes_2021_2023", "sefaz", "desoneracoes_rs_2021_2023.csv",
     "https://receitadados.sefaz.rs.gov.br/media/kbpjavgi/desoneracoes_rs_2021-2023.csv"),
    ("desoneracoes_2024", "sefaz", "desoneracoes_rs_2024.csv",
     "https://receitadados.sefaz.rs.gov.br/media/32odiq4z/desoneracoes_rs_2024.csv"),
    ("cadastro_contribuintes_setor", "sefaz", "cadastro_contribuintes_setor.csv",
     "https://receitadados.sefaz.rs.gov.br/Arquivos/Cadastro%20Contribuintes%20-%20Setor.csv"),
    ("cadastro_contribuintes_municipio", "sefaz", "cadastro_contribuintes_municipio.csv",
     "https://receitadados.sefaz.rs.gov.br/Arquivos/Cadastro%20Contribuintes%20-%20Municipio.csv"),
]

INGESTAO_TS = datetime.now().isoformat(timespec="seconds")
print("Volume:", VOL, "| ingestão:", INGESTAO_TS)

In [0]:
os.makedirs(f"{VOL}/sefaz", exist_ok=True)
log = []

for tabela, pasta, arquivo, url in FONTES_CSV:
    destino = f"{VOL}/{pasta}/{arquivo}"
    try:
        r = requests.get(url, timeout=120, verify=False, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        with open(destino, "wb") as f:
            f.write(r.content)
        log.append((tabela, arquivo, "OK", len(r.content), ""))
    except Exception as e:
        log.append((tabela, arquivo, "FALHA", 0, f"{type(e).__name__}: {str(e)[:150]}"))

display(spark.createDataFrame(log, "tabela string, arquivo string, status string, bytes long, erro string"))

In [0]:
for tabela, pasta, arquivo, _ in FONTES_CSV:
    caminho = f"{VOL}/{pasta}/{arquivo}"
    if not os.path.exists(caminho):
        continue
    bruto = open(caminho, "rb").read(600)
    encoding = "utf-8"
    try:
        texto = bruto.decode("utf-8")
    except UnicodeDecodeError:
        texto, encoding = bruto.decode("latin-1"), "latin-1"
    primeira = texto.splitlines()[0] if texto.splitlines() else ""
    sep = ";" if primeira.count(";") > primeira.count(",") else ","
    print(f"--- {tabela} | encoding={encoding} | sep='{sep}'")
    print("\n".join(texto.splitlines()[:2])[:400], "\n")

In [0]:
from pyspark.sql.functions import lit

def detectar(caminho, n=8192):
    bruto = open(caminho, "rb").read(n)
    texto = None
    # tenta remover até 3 bytes finais, que podem ser um caractere multibyte cortado
    for corte in range(4):
        pedaco = bruto[:len(bruto) - corte] if corte else bruto
        try:
            texto = pedaco.decode("utf-8")
            encoding = "UTF-8"
            break
        except UnicodeDecodeError:
            texto = None
    if texto is None:
        texto = bruto.decode("iso-8859-1")
        encoding = "ISO-8859-1"
    primeira = texto.splitlines()[0]
    sep = ";" if primeira.count(";") > primeira.count(",") else ","
    return encoding, sep

resultado = []
for tabela, pasta, arquivo, url in FONTES_CSV:
    caminho = f"{VOL}/{pasta}/{arquivo}"
    if not os.path.exists(caminho):
        resultado.append((tabela, 0, 0, "", "", "arquivo ausente")); continue
    encoding, sep = detectar(caminho)
    try:
        df = (spark.read.option("header", True).option("sep", sep)
              .option("encoding", encoding).option("quote", '"')
              .option("multiLine", False).option("inferSchema", False)
              .csv(caminho)
              .withColumn("_ingestao_ts", lit(INGESTAO_TS))
              .withColumn("_arquivo_origem", lit(arquivo))
              .withColumn("_url_origem", lit(url))
              .withColumn("_fonte", lit("SEFAZ-RS / Receita Dados")))
        df.write.mode("overwrite").option("overwriteSchema", True) \
          .saveAsTable(f"{CATALOGO}.bronze.{tabela}")
        resultado.append((tabela, df.count(), len(df.columns), encoding, sep, "OK"))
    except Exception as e:
        resultado.append((tabela, 0, 0, encoding, sep, f"{type(e).__name__}: {str(e)[:150]}"))

display(spark.createDataFrame(resultado,
    "tabela string, linhas long, colunas int, encoding string, sep string, status string"))

In [0]:
import re
from pyspark.sql.functions import lit

def limpar_nomes(df):
    """Normaliza nomes de coluna: sem espaços, acentos ou caracteres proibidos pelo Delta."""
    import unicodedata
    novos = []
    for c in df.columns:
        n = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()  # tira acentos
        n = re.sub(r"[ ,;{}()\n\t=]+", "_", n.strip())                           # troca proibidos
        n = re.sub(r"_+", "_", n).strip("_").lower()
        novos.append(n)
    return df.toDF(*novos), dict(zip(df.columns, novos))

PENDENTES = [f for f in FONTES_CSV if f[0].startswith("cadastro_contribuintes")]

for tabela, pasta, arquivo, url in PENDENTES:
    caminho = f"{VOL}/{pasta}/{arquivo}"
    encoding, sep = detectar(caminho)
    df = (spark.read.option("header", True).option("sep", sep)
          .option("encoding", encoding).option("quote", '"')
          .option("inferSchema", False).csv(caminho))
    df, mapa = limpar_nomes(df)
    df = (df.withColumn("_ingestao_ts", lit(INGESTAO_TS))
            .withColumn("_arquivo_origem", lit(arquivo))
            .withColumn("_url_origem", lit(url))
            .withColumn("_fonte", lit("SEFAZ-RS / Receita Dados")))
    df.write.mode("overwrite").option("overwriteSchema", True) \
      .saveAsTable(f"{CATALOGO}.bronze.{tabela}")
    print(f"{tabela}: {df.count()} linhas, {len(df.columns)} colunas")
    for origem, novo in mapa.items():
        print(f"   {origem!r} -> {novo}")

In [0]:
municipios = requests.get(
    "https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios",
    timeout=60).json()
with open(f"{VOL}/ibge_municipios_rs.json", "w", encoding="utf-8") as f:
    json.dump(municipios, f, ensure_ascii=False)

(spark.read.option("multiline", True).json(f"{VOL}/ibge_municipios_rs.json")
 .withColumn("_ingestao_ts", lit(INGESTAO_TS))
 .withColumn("_fonte", lit("IBGE / API Localidades"))
 .write.mode("overwrite").option("overwriteSchema", True)
 .saveAsTable(f"{CATALOGO}.bronze.ibge_municipios_rs"))

pib = requests.get(
    "https://servicodados.ibge.gov.br/api/v3/agregados/5938/periodos/-6/variaveis/37?localidades=N6[N3[43]]",
    timeout=120).json()
with open(f"{VOL}/ibge_pib_municipal_rs.json", "w", encoding="utf-8") as f:
    json.dump(pib, f, ensure_ascii=False)

(spark.read.option("multiline", True).json(f"{VOL}/ibge_pib_municipal_rs.json")
 .withColumn("_ingestao_ts", lit(INGESTAO_TS))
 .withColumn("_fonte", lit("IBGE / API Agregados tabela 5938"))
 .write.mode("overwrite").option("overwriteSchema", True)
 .saveAsTable(f"{CATALOGO}.bronze.ibge_pib_municipal_rs"))

print("Municípios:", len(municipios))

In [0]:
import pandas as pd

xlsx = f"{VOL}/cadeias/classificacao_cnae_cadeias_v2.xlsx"
pdf = pd.read_excel(xlsx, sheet_name="Classificação CNAEs", header=1, dtype=str).iloc[:, 1:]
pdf.columns = ["cnae_origem", "denominacao_cnae", "cadeia"]
pdf = pdf.dropna(subset=["cnae_origem"])

(spark.createDataFrame(pdf)
 .withColumn("_ingestao_ts", lit(INGESTAO_TS))
 .withColumn("_arquivo_origem", lit("classificacao_cnae_cadeias_v2.xlsx"))
 .withColumn("_fonte", lit("Sebrae RS - Classificação de CNAEs por cadeia produtiva 2026"))
 .write.mode("overwrite").option("overwriteSchema", True)
 .saveAsTable(f"{CATALOGO}.bronze.cnae_cadeia_produtiva"))

print("Linhas:", len(pdf))  

In [0]:
display(spark.sql(f"SHOW TABLES IN {CATALOGO}.bronze"))
for t in [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOGO}.bronze").collect()]:
    print(t, spark.table(f"{CATALOGO}.bronze.{t}").count())

In [0]:
periodos = pib[0]["resultados"][0]["series"][0]["serie"].keys()
print("Períodos do PIB:", sorted(periodos))
print("Municípios na série:", len(pib[0]["resultados"][0]["series"]))